In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib
from difflib import SequenceMatcher

matplotlib.rcParams["figure.dpi"] = 150

BASE_DIR = os.path.abspath('..')
OUT_DIR  = os.path.join(BASE_DIR, 'outputs')

df_raw = pd.read_csv(os.path.join(OUT_DIR, "ocr_batch_results.csv"))
print(f"Loaded {len(df_raw)} receipts")
print(df_raw.columns.tolist())

In [ ]:
df = df_raw.copy()

# Remove rows where total is zero or missing (zero is not a valid receipt total)
df = df[df["total_usd"].notna()]
df = df[df["total_usd"] > 0]

# Fix date
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["year"]  = df["date"].dt.year
df["month"] = df["date"].dt.month
df["month_label"] = df["date"].dt.strftime("%Y-%m")
df["weekday"] = df["date"].dt.day_name()

# Normalize category
VALID_CATEGORIES = [
    "Food & Beverage", "Grocery", "Transport", "Retail",
    "Hardware & Tools", "Electronics", "Fuel",
    "Parking", "Healthcare", "Other"
]
df["category"] = df["category"].fillna("Other").str.strip()
df.loc[~df["category"].isin(VALID_CATEGORIES), "category"] = "Other"

# Normalize payment method
PAYMENT_MAP = {
    "MASTER":      "MASTERCARD",
    "MASTERCARD":  "MASTERCARD",
    "VISA":        "VISA",
    "CASH":        "CASH",
    "DEBIT":       "DEBIT",
    "UNKNOWN":     "Unknown",
}
df["payment_method"] = (
    df["payment_method"]
    .fillna("Unknown")
    .str.upper()
    .str.strip()
    .map(lambda x: PAYMENT_MAP.get(x, "Unknown"))
)

# Normalize merchant names
df["merchant"] = df["merchant"].fillna("Unknown Merchant").str.strip().str.upper()

print(f"Valid receipts after cleaning: {len(df)}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")

In [ ]:
def fuzzy_similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

def standardize_merchants(merchants, threshold=0.75):
    """
    Groups similar merchant names together.
    e.g. 'OJO MARKETING ISDN BHD' -> 'OJC MARKETING SDN BHD'
    """
    unique = list(set(merchants))
    mapping = {}

    for name in unique:
        matched = False
        for canonical in mapping.values():
            if fuzzy_similarity(name, canonical) >= threshold:
                mapping[name] = canonical
                matched = True
                break
        if not matched:
            mapping[name] = name

    return mapping

merchant_map = standardize_merchants(df["merchant"].tolist())
df["merchant_clean"] = df["merchant"].map(merchant_map)

print("Merchant standardization sample:")
changed = {k: v for k, v in merchant_map.items() if k != v}
for original, standardized in list(changed.items())[:10]:
    print(f"  {original}  ->  {standardized}")

In [ ]:
total_raw   = len(df_raw)
total_valid = len(df)
failed      = total_raw - total_valid
zero_total  = len(df_raw[(df_raw["total_usd"].notna()) & (df_raw["total_usd"] == 0)])
unknown_pay = len(df[df["payment_method"] == "Unknown"])
other_cat   = len(df[df["category"] == "Other"])
null_merch  = len(df[df["merchant_clean"] == "UNKNOWN MERCHANT"])

print("=" * 50)
print("VISPEND AI - PIPELINE QUALITY REPORT")
print("=" * 50)
print(f"Total receipts processed   : {total_raw}")
print(f"Valid (total > 0)          : {total_valid} ({total_valid/total_raw*100:.1f}%)")
print(f"Failed / missing total     : {failed} ({failed/total_raw*100:.1f}%)")
print(f"Zero-value totals removed  : {zero_total}")
print(f"Unknown payment method     : {unknown_pay} ({unknown_pay/total_valid*100:.1f}%)")
print(f"Unclassified category      : {other_cat} ({other_cat/total_valid*100:.1f}%)")
print(f"Missing merchant           : {null_merch}")
print(f"Total spend captured       : ${df['total_usd'].sum():.2f} USD")
print(f"Average receipt value      : ${df['total_usd'].mean():.2f} USD")
print(f"Median receipt value       : ${df['total_usd'].median():.2f} USD")
print(f"Std deviation              : ${df['total_usd'].std():.2f} USD")
print("=" * 50)

In [ ]:
Q1   = df["total_usd"].quantile(0.25)
Q3   = df["total_usd"].quantile(0.75)
IQR  = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df["is_anomaly"] = (df["total_usd"] < lower_bound) | (df["total_usd"] > upper_bound)

anomalies = df[df["is_anomaly"]].sort_values("total_usd", ascending=False)

print(f"Anomalies detected (IQR method): {len(anomalies)}")
print(f"Normal spending range: ${lower_bound:.2f} - ${upper_bound:.2f} USD")
print()
print(anomalies[["file", "merchant_clean", "category", "total_usd", "date"]].to_string())

In [ ]:
cat_stats = df.groupby("category")["total_usd"].agg(
    total_spent="sum",
    avg_receipt="mean",
    median_receipt="median",
    receipt_count="count",
    std_dev="std"
).round(3).sort_values("total_spent", ascending=False)

print("SPENDING BY CATEGORY (USD):")
print(cat_stats.to_string())

# Monthly trend
monthly = df.dropna(subset=["date"]).groupby("month_label")["total_usd"].agg(
    total="sum",
    count="count",
    average="mean"
).round(3).sort_index()

print("\nMONTHLY SPENDING TREND:")
print(monthly.to_string())

# Weekday analysis
weekday_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
weekday_spend = df.dropna(subset=["weekday"]).groupby("weekday")["total_usd"].agg(
    total="sum", count="count"
).reindex(weekday_order).dropna()

print("\nSPENDING BY WEEKDAY:")
print(weekday_spend.to_string())

In [ ]:
merchant_stats = df.groupby("merchant_clean")["total_usd"].agg(
    total_spent="sum",
    visit_count="count",
    avg_per_visit="mean"
).round(3).sort_values("total_spent", ascending=False)

print("TOP 15 MERCHANTS BY SPEND:")
print(merchant_stats.head(15).to_string())

In [ ]:
# Spending tiers
def spending_tier(amount):
    if amount < 5:
        return "Micro (< $5)"
    elif amount < 20:
        return "Small ($5-$20)"
    elif amount < 50:
        return "Medium ($20-$50)"
    elif amount < 100:
        return "Large ($50-$100)"
    else:
        return "High (> $100)"

df["spending_tier"] = df["total_usd"].apply(spending_tier)

tier_order = ["Micro (< $5)", "Small ($5-$20)", "Medium ($20-$50)", "Large ($50-$100)", "High (> $100)"]
tier_counts = df["spending_tier"].value_counts().reindex(tier_order).fillna(0)

print("SPENDING TIER DISTRIBUTION:")
print(tier_counts.to_string())

In [ ]:
fig_dir = OUT_DIR
colors  = ["#0077B6", "#00B4D8", "#48CAE4", "#90E0EF", "#ADE8F4", "#CAF0F8", "#023E8A", "#0096C7"]

# Chart 1 — Spending by category (bar)
cat_spend = df.groupby("category")["total_usd"].sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(cat_spend.index, cat_spend.values, color=colors[:len(cat_spend)], edgecolor="white")
ax.set_title("Total Spending by Category (USD)", fontsize=14, fontweight="bold")
ax.set_xlabel("Category")
ax.set_ylabel("Total Spend (USD)")
plt.xticks(rotation=30, ha="right")
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f"${bar.get_height():.0f}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "chart_01_category_bar.png"), dpi=150)
plt.show()

# Chart 2 — Category pie
fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(cat_spend.values, labels=cat_spend.index, autopct="%1.1f%%",
       colors=colors, startangle=140, wedgeprops={"edgecolor": "white"})
ax.set_title("Spending Distribution by Category", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "chart_02_category_pie.png"), dpi=150)
plt.show()

# Chart 3 — Payment methods
pm = df["payment_method"].value_counts()
fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(pm.values, labels=pm.index, autopct="%1.1f%%",
       colors=colors, startangle=140, wedgeprops={"edgecolor": "white"})
ax.set_title("Payment Methods Used", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "chart_03_payment_methods.png"), dpi=150)
plt.show()

# Chart 4 — Monthly spending trend
df_time = df.dropna(subset=["month_label"])
monthly_total = df_time.groupby("month_label")["total_usd"].sum().sort_index()
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(monthly_total.index, monthly_total.values,
        color="#0077B6", linewidth=2.5, marker="o", markersize=6)
ax.fill_between(monthly_total.index, monthly_total.values, alpha=0.12, color="#0077B6")
ax.set_title("Monthly Spending Trend (USD)", fontsize=14, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Total Spend (USD)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "chart_04_monthly_trend.png"), dpi=150)
plt.show()

# Chart 5 — Top 10 merchants
top_m = df.groupby("merchant_clean")["total_usd"].sum().sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(top_m.index[::-1], top_m.values[::-1], color="#00B4D8")
ax.set_title("Top 10 Merchants by Total Spend (USD)", fontsize=14, fontweight="bold")
ax.set_xlabel("Total Spend (USD)")
for bar in bars:
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f"${bar.get_width():.1f}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "chart_05_top_merchants.png"), dpi=150)
plt.show()

# Chart 6 — Spending tier distribution
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(tier_counts.index, tier_counts.values, color=colors[:5], edgecolor="white")
ax.set_title("Receipt Spending Tier Distribution", fontsize=14, fontweight="bold")
ax.set_xlabel("Spending Tier")
ax.set_ylabel("Number of Receipts")
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
            int(bar.get_height()), ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "chart_06_spending_tiers.png"), dpi=150)
plt.show()

# Chart 7 — Anomaly flag chart
fig, ax = plt.subplots(figsize=(12, 4))
normal    = df[~df["is_anomaly"]]
anomalous = df[df["is_anomaly"]]
ax.scatter(normal.index, normal["total_usd"], color="#00B4D8", label="Normal", s=30, alpha=0.7)
ax.scatter(anomalous.index, anomalous["total_usd"], color="#EF233C", label="Anomaly", s=60, zorder=5)
ax.axhline(upper_bound, color="#EF233C", linestyle="--", linewidth=1, label=f"Upper bound (${upper_bound:.1f})")
ax.set_title("Receipt Spend — Anomaly Detection (IQR Method)", fontsize=14, fontweight="bold")
ax.set_ylabel("Total USD")
ax.set_xlabel("Receipt Index")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "chart_07_anomaly_detection.png"), dpi=150)
plt.show()

# Chart 8 — Weekday spending
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(weekday_spend.index, weekday_spend["total"], color="#0077B6", edgecolor="white")
ax.set_title("Total Spending by Day of Week (USD)", fontsize=14, fontweight="bold")
ax.set_xlabel("Weekday")
ax.set_ylabel("Total Spend (USD)")
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"${bar.get_height():.0f}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "chart_08_weekday_spend.png"), dpi=150)
plt.show()

print("All 8 charts saved to outputs/")

In [ ]:
# Final enriched CSV
df.drop(columns=["items", "ocr_text_preview"], errors="ignore").to_csv(
    os.path.join(OUT_DIR, "final_analytics.csv"), index=False
)

# Category summary
cat_stats.to_csv(os.path.join(OUT_DIR, "category_summary.csv"))

# Monthly summary
monthly.to_csv(os.path.join(OUT_DIR, "monthly_summary.csv"))

# Anomalies
anomalies[["file", "merchant_clean", "category", "total_usd", "date"]].to_csv(
    os.path.join(OUT_DIR, "anomalies.csv"), index=False
)

# Merchant summary
merchant_stats.head(20).to_csv(os.path.join(OUT_DIR, "merchant_summary.csv"))

print("All outputs saved:")
print("  final_analytics.csv")
print("  category_summary.csv")
print("  monthly_summary.csv")
print("  anomalies.csv")
print("  merchant_summary.csv")
print("  financial_insights.txt")
print("  8 chart PNG files")